# Working With APIs

A great source for data is APIs. API means Application Programming Interface, which is somewhat misleading because the term was used originally for software libraries. But now it is commonly used to refer to web APIs. Web APIs are interfaces that allow you to interact with a web service and access its data. They are usually based on HTTP and use JSON or XML as the data format. You can use APIs to get data from various sources, such as social media, weather, news, etc. You can also use APIs to send data to a web service, such as posting a tweet or creating a new issue on GitHub. To work with APIs in Python, you can use the `requests` library, which provides a simple and elegant way to make HTTP requests.

## Setup

Before you can start you maybe need to install some packages if not already done.

In [ ]:
%pip install -r ../requirements.txt

import requests # Working with HTTP requests
import json # Working with JSON data
import csv # Working with CSV files

## Requests

Let's get some data from an API. If you need an API to play with, you can look at [public-apis/public-apis: A collective list of free APIs](https://github.com/public-apis/public-apis). Some need authentication but a lot of them are free to use without authentication.

To request data from an API you need the URL of the endpoint and maybe some parameters.

In [ ]:
url = "https://uselessfacts.jsph.pl/api/v2/facts/random"

response = requests.get(url)

try:
    response.raise_for_status()
except requests.exceptions.HTTPError as e:
    print(f"Fehler beim laden der Resource: {e}")

useless_fact = json.loads(response.text)

# Pretty print the JSON object
print(json.dumps(useless_fact, indent=4))

Sure. There are more useful examples than this one. Here is an example with authentication and parameters. It requests open jobs from germanies official job board and writes them into a CSV file.

The API's documentation can be found here: [bundesAPI/jobsuche-api: API zur Bundesagentur für Arbeit Jobsuche](https://github.com/bundesAPI/jobsuche-api)

In [ ]:
# Here are some parameters you can adjust to get different results. You can find more parameters in the API's documentation.
job_name = ""
job_location = "Essen"
job_professional_field = "Informatik"
job_published_since = 30
job_radius = 25
page = 1
size = 50

# Creates the url for the request from the parameters.
url = f"https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v4/jobs?wo={job_location}{f'&was={job_name}' if job_name else ''}{f'&berufsfeld={job_professional_field}' if job_professional_field else ''}&page={page}&size={size}&veroeffentlichtseit={job_published_since}&zeitarbeit=false&angebotsart=1&befristung=2&arbeitszeit=vz&umkreis={job_radius}"

# Sends the request. The API key is part of the headers.
response = requests.get(url, headers={"accept": "application/json", "X-API-Key": "jobboerse-jobsuche"})

try:
    response.raise_for_status()
except requests.exceptions.HTTPError as e:
    print(f"Fehler beim laden der Resource: {e}")

job_data = json.loads(response.text)

# Converts the resulting dictionary into a list of lists, which can be easily written to a CSV file.
data = [['Job', 'Title', 'Ref-Nr.', 'Street', 'Zip Code', 'City', 'Region', 'Distance', 'Company', 'Published', 'URL']]

for job in job_data["stellenangebote"]:
    data.append([
        job.get("beruf", ""),
        job.get("titel", ""),
        job.get("refnr", ""),
        job.get("strasse", ""),
        job.get("plz", ""),
        job.get("ort", ""),
        job.get("region", ""),
        job.get("entfernung", ""),
        job.get("arbeitgeber", ""),
        job.get("aktuelleVeroeffentlichungsdatum", ""),
        job.get("externeUrl", "")])

# Writes the data to a CSV file.
with open('../temp/jobs.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file, quotechar="\"", delimiter=',', quoting=csv.QUOTE_MINIMAL)
    writer.writerows(data)

# You can look into /temp/jobs.csv to see the results.
